In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
current_dir = "/home/nfash/EECS_545"

# List all files in the current directory (excluding subdirectories)
files_in_dir = [f for f in os.listdir(current_dir)]
for file in files_in_dir:
    print(file)

    
# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

catboost_info
.ipynb_checkpoints
isic-2024-challenge.zip
tree-based-models.ipynb
cnn-based-models.ipynb
testdir
isic-2024-challenge
metadata_model.ipynb
train-metadata-preprocessed-tree-model.csv
test-metadata-preprocessed-tree-model.csv


## Libraries and dependencies

In [2]:
import numpy as np 
import pandas as pd 
import os 
import h5py
import cv2
#from google.colab.patches import cv2_imshow
from IPython.display import display, Javascript
import plotly.express as px
from sklearn.model_selection import train_test_split,cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer,KNNImputer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, FunctionTransformer
from sklearn.feature_extraction import FeatureHasher
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import accuracy_score, mean_squared_error,get_scorer_names
from sklearn.preprocessing import LabelEncoder,MinMaxScaler
from sklearn.linear_model import LogisticRegression, LinearRegression,SGDClassifier,BayesianRidge
from sklearn.utils import resample
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from imblearn.combine import *
from imblearn.under_sampling import *
from imblearn.over_sampling import *
from imblearn.pipeline import Pipeline
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import xgboost as xgb
from sklearn.model_selection import cross_val_score
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
import catboost as cb
from sklearn.ensemble import VotingClassifier,StackingClassifier
import matplotlib.pyplot as plt

In [3]:
np.__version__

'1.23.5'

In [4]:
# Deep learning keras libraries
import keras
print(keras.__version__)
import tensorflow as tf

3.5.0


## Data loading


In [5]:
current_dir = "/home/nfash/EECS_545/isic-2024-challenge"
sample_submission = pd.read_csv(os.path.join(current_dir,'sample_submission.csv'))
sample_submission.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3 entries, 0 to 2
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   isic_id  3 non-null      object 
 1   target   3 non-null      float64
dtypes: float64(1), object(1)
memory usage: 180.0+ bytes


In [6]:
sample_submission.head()

,isic_id,target
0,ISIC_0015657,0.3
1,ISIC_0015729,0.3
2,ISIC_0015740,0.3


In [7]:
train_metadata = pd.read_csv(os.path.join(current_dir,'train-metadata.csv'),low_memory=False)
#train_metadata.info()

In [8]:
train_metadata.head()

,isic_id,target,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,...,lesion_id,iddx_full,iddx_1,iddx_2,iddx_3,iddx_4,iddx_5,mel_mitotic_index,mel_thick_mm,tbp_lv_dnn_lesion_confidence
0,ISIC_0015670,0,IP_1235828,60.0,male,lower extremity,3.04,TBP tile: close-up,3D: white,20.244422,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,97.517282
1,ISIC_0015845,0,IP_8170065,60.0,male,head/neck,1.10,TBP tile: close-up,3D: white,31.712570,...,IL_6727506,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,3.141455
2,ISIC_0015864,0,IP_6724798,60.0,male,posterior torso,3.40,TBP tile: close-up,3D: XP,22.575830,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.804040
3,ISIC_0015902,0,IP_4111386,65.0,male,anterior torso,3.22,TBP tile: close-up,3D: XP,14.242329,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,99.989998
4,ISIC_0024200,0,IP_8313778,55.0,male,anterior torso,2.73,TBP tile: close-up,3D: white,24.725520,...,NaN,Benign,Benign,NaN,NaN,NaN,NaN,NaN,NaN,70.442510


In [9]:
test_metadata = pd.read_csv(os.path.join(current_dir,'test-metadata.csv'),low_memory=False)
#test_metadata.info()

In [10]:
test_metadata.head()

,isic_id,patient_id,age_approx,sex,anatom_site_general,clin_size_long_diam_mm,image_type,tbp_tile_type,tbp_lv_A,tbp_lv_Aext,...,tbp_lv_radial_color_std_max,tbp_lv_stdL,tbp_lv_stdLExt,tbp_lv_symm_2axis,tbp_lv_symm_2axis_angle,tbp_lv_x,tbp_lv_y,tbp_lv_z,attribution,copyright_license
0,ISIC_0015657,IP_6074337,45.0,male,posterior torso,2.70,TBP tile: close-up,3D: XP,22.80433,20.007270,...,0.304827,1.281532,2.299935,0.479339,20,-155.06510,1511.222000,113.980100,Memorial Sloan Kettering Cancer Center,CC-BY
1,ISIC_0015729,IP_1664139,35.0,female,lower extremity,2.52,TBP tile: close-up,3D: XP,16.64867,9.657964,...,0.000000,1.271940,2.011223,0.426230,25,-112.36924,629.535889,-15.019287,"Frazer Institute, The University of Queensland...",CC-BY
2,ISIC_0015740,IP_7142616,65.0,male,posterior torso,3.16,TBP tile: close-up,3D: XP,24.25384,19.937380,...,0.230742,1.080308,2.705857,0.366071,110,-84.29282,1303.978000,-28.576050,FNQH Cairns,CC-BY


#### Load Image Byte String

In this competition, images are provided as byte strings. The following code snippet demonstrates how to load these images into memory. One might wonder why the provided jpeg images aren't being used in the /train-image folder for training. This is because testing images are not provided as JPEG images; instead, they are provided as byte strings. Why use byte strings? They occupy significantly less memory compared to np.array representations.


In [11]:
# image sample visualization
training_validation_hdf5 = h5py.File(f"{current_dir}/train-image.hdf5", 'r')
testing_hdf5 = h5py.File(f"{current_dir}/test-image.hdf5", 'r')

In [12]:
# sample randomly two images from the train dataset
img_sample = train_metadata['isic_id'].sample(n=2).to_list()

# load the image from byte arrays, 
byte_str = [training_validation_hdf5[isic_id][()] for isic_id in img_sample]

# convert byte str to numpy array
img_arr = [np.frombuffer(byte, np.uint8) for byte in byte_str]

# convert cv2 image
img_cv2 = [cv2.imdecode(nparr, cv2.IMREAD_COLOR) for nparr in img_arr] 
for ind,val in enumerate(img_cv2):
    print(f"Image {img_sample[ind]}:")
    print(f"Shape:{val.shape}")
    #cv2_imshow(val)

Image ISIC_1291355:
Shape:(119, 119, 3)
Image ISIC_4776899:
Shape:(125, 125, 3)


In [13]:
# sample randomly two images from the train dataset
img_sample = test_metadata['isic_id'].sample(n=3).to_list()

# load the image from byte arrays, 
byte_str = [testing_hdf5[isic_id][()] for isic_id in img_sample]

# convert byte str to numpy array
img_arr = [np.frombuffer(byte, np.uint8) for byte in byte_str]

# convert cv2 image
img_cv2 = [cv2.imdecode(nparr, cv2.IMREAD_COLOR) for nparr in img_arr] 
for ind,val in enumerate(img_cv2):
    print(f"Image {img_sample[ind]}:")
    print(f"Shape:{val.shape}")
    #cv2_imshow(val)

Image ISIC_0015740:
Shape:(119, 119, 3)
Image ISIC_0015657:
Shape:(141, 141, 3)
Image ISIC_0015729:
Shape:(125, 125, 3)


## Preprocessing

In [14]:
class DataPreprocessor:
    def __init__(self):
        self.preprocessor = None
        self.train_columns = None

    def fit_transform(self, df):
        """Preprocess training data and store transformations."""
        df = df.copy()
        df = self._drop_irrelevant_columns(df)
        df = self._drop_train_only_columns(df)
        categorical_cols, numerical_cols = self._identify_column_types(df)
        
        # Define preprocessing pipelines
        numerical_pipeline = Pipeline([
            # ("imputer", SimpleImputer(strategy="median")),
            ("imputer", KNNImputer()),
            ("scaler", StandardScaler())
        ])
        
        categorical_pipeline = Pipeline([
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ])
        
        self.preprocessor = ColumnTransformer([
            ("num", numerical_pipeline, numerical_cols),
            ("cat", categorical_pipeline, categorical_cols)
        ])
        
        transformed_data = self.preprocessor.fit_transform(df)
        
        cat_feature_names = self.preprocessor.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_cols)
        all_columns = numerical_cols + list(cat_feature_names)
        
        df_processed = pd.DataFrame(transformed_data, columns=all_columns)
        df_processed["isic_id"] = df["isic_id"].values
        df_processed["target"] = df["target"].values
        
        self.train_columns = df_processed.columns  # Store train columns
        return df_processed

    def transform(self, df):
        """Preprocess test data using stored transformations from training."""
        df = df.copy()
        df = self._drop_irrelevant_columns(df)
        
        transformed_data = self.preprocessor.transform(df)
        cat_feature_names = self.preprocessor.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out()
        all_columns = self.train_columns[:-2]  # Exclude 'isic_id' and 'target'
        
        df_processed = pd.DataFrame(transformed_data, columns=all_columns)
        df_processed["isic_id"] = df["isic_id"].values
        # df_processed["target"] = df["target"].values
        
        # Align test dataset with train columns
        df_processed = self._align_train_test_columns(df_processed)
        return df_processed

    def _drop_irrelevant_columns(self, df):
        """Remove unnecessary columns."""
        return df.drop(columns=['patient_id','image_type', 'tbp_tile_type', 'attribution', 'copyright_license'], errors="ignore")

    def _drop_train_only_columns(self,df):
        """Remove columns that are present only in the train set and not in the test set"""
        drop_train_only_columns = [
            'lesion_id', 'iddx_full', 'iddx_1', 'iddx_2', 'iddx_3', 'iddx_4', 'iddx_5',
            'mel_mitotic_index', 'mel_thick_mm', 'tbp_lv_dnn_lesion_confidence'
            ]
        return df.drop(columns=drop_train_only_columns,errors='ignore')
        

    def _identify_column_types(self, df):
        """Identify categorical and numerical columns, excluding 'isic_id' and 'target'."""
        categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
        numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
        categorical_cols = [col for col in categorical_cols if col != "isic_id"]
        numerical_cols = [col for col in numerical_cols if col != "target"]
        return categorical_cols, numerical_cols
    
    def _align_train_test_columns(self, df):
        """Ensure test data has the same columns as train data."""
        train_cols = list(self.train_columns)
        train_cols.remove('target')
        missing_cols = set(train_cols) - set(df.columns)
        for col in missing_cols:
            df[col] = 0
        return df[train_cols]


In [15]:
train_metadata = pd.read_csv(os.path.join(current_dir, 'train-metadata.csv'), low_memory=False)
test_metadata = pd.read_csv(os.path.join(current_dir, 'test-metadata.csv'), low_memory=False)

In [38]:
data_object = DataPreprocessor() #SLOW
#train_metadata_processed = data_object.fit_transform(train_metadata)
#test_metadata_processed = data_object.transform(test_metadata)

train_metadata_processed = pd.read_csv('train-metadata-preprocessed-tree-model.csv',low_memory=False)
test_metadata_processed = pd.read_csv('test-metadata-preprocessed-tree-model.csv',low_memory=False)

(401059, 74)

In [39]:
# Save the preprocessed data
#train_metadata_processed.to_csv('train-metadata-preprocessed-tree-model.csv')
#test_metadata_processed.to_csv('test-metadata-preprocessed-tree-model.csv')

In [40]:
X_train = train_metadata_processed.drop(columns=['target'])
y_train = train_metadata_processed['target']
X_test = test_metadata_processed.copy()


## handling class imbalance

In [41]:
def resampler_data(X, Y):
    
    # Apply undersampling only on numerical data
    resampler = RandomUnderSampler(sampling_strategy=0.1,)
    X_resampled, Y_resampled = resampler.fit_resample(X,Y)
    
    # now apply over sampling for the minorit class
    resampler = RandomOverSampler(sampling_strategy=0.3)
    X_resampled, Y_resampled = resampler.fit_resample(X_resampled, Y_resampled)

    print("X_final shape;",X_resampled.shape)
    print("Y_final shape:",Y_resampled.shape)
    
    return X_resampled, Y_resampled

In [42]:
X_train_final, y_train_final = resampler_data(X_train, y_train)

/home/nfash/miniconda3/envs/keras_env/lib/python3.11/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/home/nfash/miniconda3/envs/keras_env/lib/python3.11/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


X_final shape; (5109, 73)
Y_final shape: (5109,)


/home/nfash/miniconda3/envs/keras_env/lib/python3.11/site-packages/sklearn/base.py:484: FutureWarning: `BaseEstimator._check_n_features` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_n_features` instead.
  warnings.warn(
/home/nfash/miniconda3/envs/keras_env/lib/python3.11/site-packages/sklearn/base.py:493: FutureWarning: `BaseEstimator._check_feature_names` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation._check_feature_names` instead.
  warnings.warn(


## metadata only model for submission checking

In [43]:
# Compute class weights
class_weights = y_train_final.value_counts(normalize=True).to_dict()
class_weights = {k: 1/v for k, v in class_weights.items()}
print(class_weights)

{0: 1.2999999999999998, 1: 4.333333333333333}


In [44]:
# Convert data to DMatrix for XGBoost
dtrain = xgb.DMatrix(X_train_final.drop(columns=['isic_id'],axis=1), label=y_train_final, 
                     weight=[class_weights[y] for y in y_train_final])
# dval = xgb.DMatrix(X_val.drop(columns=['isic_id'],axis=1), label=y_val)

In [45]:
# Define XGBoost parameters
params = {
    'objective': 'binary:logistic',
    'eval_metric': 'logloss',
    'scale_pos_weight': class_weights[1] / class_weights[0],
    'max_depth': 10,
    'eta': 0.01,
    'subsample': 0.9,
    'colsample_bytree': 0.9,
    'random_state': 63
}

# # Train the XGBoost classifier with cross-validation
# cv_results = xgb.cv(
#     params,
#     dtrain,
#     num_boost_round=10000,
#     nfold=5,
#     early_stopping_rounds=10,
#     metrics='logloss',
#     as_pandas=True,
#     seed=42
# )

In [46]:
# # View the train set results
# print("Cross-validation results:")
# print(cv_results)

In [47]:
# num_boost_rounds = cv_results['test-logloss-mean'].idxmin()
# xgb_model = xgb.train(params, dtrain, num_boost_round=num_boost_rounds)
# print(num_boost_rounds)

In [48]:
# light gbm model
lgb_matrix = lgb.Dataset(X_train_final.drop(columns=['isic_id'],axis=1), label=y_train_final, weight=[class_weights[y] for y in y_train_final])

lgb_params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'eta': 0.01,
    'num_iterations':10000,
    'num_leaves':63,
    'random_state': 42,
    'verbose':-1
}

In [49]:
# lgb_model = lgb.train(lgb_params, lgb_matrix, num_boost_round=1000)

In [50]:
# # CatBoost model
# cb_model = cb.CatBoostClassifier(iterations=10000, depth=6, learning_rate=0.01, loss_function='Logloss', random_seed=42)
# cb_model.fit(X_train_final.drop(columns=['isic_id'],axis=1), y_train_final, sample_weight=[class_weights[y] for y in y_train_final], verbose=0)


In [51]:
# Predict class probabilities on the test data
dtest = xgb.DMatrix(X_test.drop(columns=['isic_id'],axis=1))
# y_test_pred_proba = model.predict(dtest)
# # Output the predicted probabilities
# print(y_test_pred_proba)

In [52]:
xgb_pred = lambda X: xgb_model.predict(dtest)
lgb_pred = lambda X: lgb_model.predict(X_test.drop(columns=['isic_id'],axis=1))
cb_pred = lambda X: cb_model.predict_proba(X_test.drop(columns=['isic_id'],axis=1))[:, 1]

In [53]:
# xgb_pred(dtest)

In [54]:
# lgb_pred(X_test)

In [55]:
# cb_pred(X_test)

In [56]:
# final_pred = np.mean([xgb_pred(dtest),lgb_pred(X_test),cb_pred(X_test)],axis=0)
# final_pred

In [83]:
# Create Stacking Model
stacking_model = StackingClassifier(
    estimators=[
        ('xgb', xgb.XGBClassifier(**params)),
        ('lgb', lgb.LGBMClassifier(**lgb_params)),
        ('cb', cb.CatBoostClassifier(iterations=10000,depth=10, learning_rate=0.01, 
                                     loss_function='Logloss', random_seed=42))
    ],
    final_estimator=LogisticRegression(),
    stack_method='predict_proba',
    verbose=2,
    n_jobs=-1,
    cv = 5
)

In [84]:
stacking_model.fit(X_train_final.drop(columns=['isic_id'],axis=1), y_train_final)
final_pred = stacking_model.predict_proba(X_test.drop(columns=['isic_id'],axis=1))

TBB Warning: The number of workers is currently limited to 15. The request for 35 workers is ignored. Further requests for more workers will be silently ignored until the limit changes.



0:	learn: 0.6803878	total: 73.2ms	remaining: 12m 12s
1:	learn: 0.6680535	total: 128ms	remaining: 10m 39s
2:	learn: 0.6557396	total: 174ms	remaining: 9m 39s
3:	learn: 0.6442398	total: 270ms	remaining: 11m 13s
4:	learn: 0.6337820	total: 344ms	remaining: 11m 28s
5:	learn: 0.6229353	total: 387ms	remaining: 10m 45s
6:	learn: 0.6118546	total: 467ms	remaining: 11m 6s
7:	learn: 0.6006781	total: 502ms	remaining: 10m 26s
8:	learn: 0.5890460	total: 556ms	remaining: 10m 17s
9:	learn: 0.5779352	total: 579ms	remaining: 9m 37s
10:	learn: 0.5704045	total: 648ms	remaining: 9m 48s
11:	learn: 0.5610709	total: 704ms	remaining: 9m 46s
12:	learn: 0.5511940	total: 742ms	remaining: 9m 30s
13:	learn: 0.5420906	total: 786ms	remaining: 9m 20s
14:	learn: 0.5326233	total: 826ms	remaining: 9m 10s
15:	learn: 0.5236715	total: 892ms	remaining: 9m 16s
16:	learn: 0.5148699	total: 962ms	remaining: 9m 24s
17:	learn: 0.5068862	total: 1.05s	remaining: 9m 44s
18:	learn: 0.4982475	total: 1.08s	remaining: 9m 28s
19:	learn: 0.4

[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 16 concurrent workers.
TBB Warning: The number of workers is currently limited to 15. The request for 35 workers is ignored. Further requests for more workers will be silently ignored until the limit changes.



0:	learn: 0.6817439	total: 212ms	remaining: 35m 17s
0:	learn: 0.6813822	total: 280ms	remaining: 46m 44s
0:	learn: 0.6817888	total: 300ms	remaining: 49m 57s
0:	learn: 0.6812625	total: 105ms	remaining: 17m 34s
1:	learn: 0.6695413	total: 364ms	remaining: 30m 17s
0:	learn: 0.6801470	total: 208ms	remaining: 34m 34s
1:	learn: 0.6697312	total: 463ms	remaining: 38m 34s
1:	learn: 0.6683263	total: 476ms	remaining: 39m 39s
1:	learn: 0.6694501	total: 251ms	remaining: 20m 52s
2:	learn: 0.6579010	total: 501ms	remaining: 27m 49s
1:	learn: 0.6686978	total: 325ms	remaining: 27m 5s
2:	learn: 0.6582874	total: 605ms	remaining: 33m 34s
2:	learn: 0.6574239	total: 614ms	remaining: 34m 6s
2:	learn: 0.6581006	total: 367ms	remaining: 20m 24s
3:	learn: 0.6466555	total: 640ms	remaining: 26m 40s
2:	learn: 0.6560778	total: 467ms	remaining: 25m 57s
3:	learn: 0.6460333	total: 714ms	remaining: 29m 44s
3:	learn: 0.6468479	total: 476ms	remaining: 19m 48s
3:	learn: 0.6464700	total: 771ms	remaining: 32m 7s
3:	learn: 0.644

[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed:   10.7s finished


...

9785:	learn: 0.0013919	total: 15m 59s	remaining: 21s
9599:	learn: 0.0014784	total: 15m 59s	remaining: 40s
9593:	learn: 0.0013312	total: 16m	remaining: 40.6s
8883:	learn: 0.0013519	total: 16m	remaining: 2m
9786:	learn: 0.0013919	total: 15m 59s	remaining: 20.9s
9320:	learn: 0.0015932	total: 16m	remaining: 1m 9s
9600:	learn: 0.0014784	total: 15m 59s	remaining: 39.9s
8884:	learn: 0.0013519	total: 16m	remaining: 2m
9594:	learn: 0.0013312	total: 16m	remaining: 40.5s
9787:	learn: 0.0013919	total: 15m 59s	remaining: 20.8s
9321:	learn: 0.0015932	total: 16m	remaining: 1m 9s
8885:	learn: 0.0013519	total: 16m	remaining: 2m
9601:	learn: 0.0014784	total: 15m 59s	remaining: 39.8s
9595:	learn: 0.0013312	total: 16m	remaining: 40.4s
9788:	learn: 0.0013919	total: 16m	remaining: 20.7s
8886:	learn: 0.0013519	total: 16m	remaining: 2m
9322:	learn: 0.0015932	total: 16m	remaining: 1m 9s
9602:	learn: 0.0014784	total: 16m	remaining: 39.7s
9789:	learn: 0.0013919	total: 16m	remaining: 20.6s
8887:	learn: 0.0013519	

9377:	learn: 0.0015925	total: 16m 6s	remaining: 1m 4s
9849:	learn: 0.0013905	total: 16m 5s	remaining: 14.7s
9659:	learn: 0.0014780	total: 16m 5s	remaining: 34s
8944:	learn: 0.0013517	total: 16m 6s	remaining: 1m 53s
9656:	learn: 0.0013308	total: 16m 6s	remaining: 34.3s
9660:	learn: 0.0014780	total: 16m 5s	remaining: 33.9s
9850:	learn: 0.0013905	total: 16m 6s	remaining: 14.6s
9378:	learn: 0.0015925	total: 16m 6s	remaining: 1m 3s
9657:	learn: 0.0013308	total: 16m 6s	remaining: 34.2s
9851:	learn: 0.0013904	total: 16m 6s	remaining: 14.5s
9661:	learn: 0.0014780	total: 16m 6s	remaining: 33.8s
8945:	learn: 0.0013517	total: 16m 6s	remaining: 1m 53s
9379:	learn: 0.0015925	total: 16m 6s	remaining: 1m 3s
9658:	learn: 0.0013308	total: 16m 6s	remaining: 34.1s
9852:	learn: 0.0013904	total: 16m 6s	remaining: 14.4s
9662:	learn: 0.0014780	total: 16m 6s	remaining: 33.7s
9380:	learn: 0.0015925	total: 16m 6s	remaining: 1m 3s
8946:	learn: 0.0013517	total: 16m 6s	remaining: 1m 53s
9659:	learn: 0.0013308	tota

9716:	learn: 0.0013287	total: 16m 12s	remaining: 28.3s
9721:	learn: 0.0014776	total: 16m 11s	remaining: 27.8s
9912:	learn: 0.0013900	total: 16m 11s	remaining: 8.53s
8998:	learn: 0.0013516	total: 16m 12s	remaining: 1m 48s
9439:	learn: 0.0015913	total: 16m 12s	remaining: 57.7s
9717:	learn: 0.0013287	total: 16m 12s	remaining: 28.2s
9722:	learn: 0.0014776	total: 16m 12s	remaining: 27.7s
9913:	learn: 0.0013900	total: 16m 12s	remaining: 8.43s
8999:	learn: 0.0013516	total: 16m 12s	remaining: 1m 48s
9440:	learn: 0.0015913	total: 16m 12s	remaining: 57.6s
9718:	learn: 0.0013287	total: 16m 12s	remaining: 28.1s
9723:	learn: 0.0014776	total: 16m 12s	remaining: 27.6s
9914:	learn: 0.0013900	total: 16m 12s	remaining: 8.33s
9000:	learn: 0.0013516	total: 16m 12s	remaining: 1m 47s
9441:	learn: 0.0015913	total: 16m 12s	remaining: 57.5s
9719:	learn: 0.0013287	total: 16m 12s	remaining: 28s
9724:	learn: 0.0014776	total: 16m 12s	remaining: 27.5s
9915:	learn: 0.0013900	total: 16m 12s	remaining: 8.24s
9001:	lea

9776:	learn: 0.0013284	total: 16m 18s	remaining: 22.3s
9497:	learn: 0.0015908	total: 16m 18s	remaining: 51.7s
9054:	learn: 0.0013515	total: 16m 18s	remaining: 1m 42s
9975:	learn: 0.0013886	total: 16m 17s	remaining: 2.35s
9780:	learn: 0.0014755	total: 16m 17s	remaining: 21.9s
9777:	learn: 0.0013284	total: 16m 18s	remaining: 22.2s
9498:	learn: 0.0015908	total: 16m 18s	remaining: 51.6s
9055:	learn: 0.0013515	total: 16m 18s	remaining: 1m 41s
9976:	learn: 0.0013886	total: 16m 18s	remaining: 2.25s
9781:	learn: 0.0014755	total: 16m 18s	remaining: 21.8s
9499:	learn: 0.0015908	total: 16m 18s	remaining: 51.5s
9778:	learn: 0.0013284	total: 16m 18s	remaining: 22.1s
9782:	learn: 0.0014755	total: 16m 18s	remaining: 21.7s
9056:	learn: 0.0013515	total: 16m 18s	remaining: 1m 41s
9977:	learn: 0.0013886	total: 16m 18s	remaining: 2.16s
9779:	learn: 0.0013284	total: 16m 18s	remaining: 22s
9783:	learn: 0.0014755	total: 16m 18s	remaining: 21.6s
9500:	learn: 0.0015908	total: 16m 18s	remaining: 51.4s
9978:	lea

[Parallel(n_jobs=-1)]: Done   3 out of   5 | elapsed: 16.7min remaining: 11.1min
[Parallel(n_jobs=-1)]: Done   5 out of   5 | elapsed: 17.1min finished


In [85]:
#final_pred = final_pred[:,1]
final_pred

array([[0.99807534, 0.00192466],
       [0.99809975, 0.00190025],
       [0.99804961, 0.00195039]])

In [87]:
#StackingClassifier?

In [101]:
train_metadata_processed.head()

,age_approx,clin_size_long_diam_mm,tbp_lv_A,tbp_lv_Aext,tbp_lv_B,tbp_lv_Bext,tbp_lv_C,tbp_lv_Cext,tbp_lv_H,tbp_lv_Hext,...,tbp_lv_location_simple_Head & Neck,tbp_lv_location_simple_Left Arm,tbp_lv_location_simple_Left Leg,tbp_lv_location_simple_Right Arm,tbp_lv_location_simple_Right Leg,tbp_lv_location_simple_Torso Back,tbp_lv_location_simple_Torso Front,tbp_lv_location_simple_Unknown,isic_id,target
0,0.146694,-0.511069,0.067613,0.380443,-0.257500,-0.659882,-0.192995,-0.407543,-0.288931,-0.917620,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,ISIC_0015670,0
1,0.146694,-1.624050,2.935019,2.959583,-0.369545,-0.527265,1.126869,0.906538,-2.708061,-3.006591,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,ISIC_0015845,0
2,0.146694,-0.304536,0.650540,0.625867,1.835454,1.466075,1.644677,1.385391,0.835409,0.339647,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,ISIC_0015864,0
3,0.515280,-0.407803,-1.433104,-0.780446,-1.294561,-1.291919,-1.583639,-1.355725,0.318926,-0.166282,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,ISIC_0015902,0
4,-0.221891,-0.688917,1.188031,1.455843,-0.344179,-0.268249,0.250792,0.349419,-1.396095,-1.590171,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,ISIC_0024200,0


In [88]:
sub_df = pd.concat([X_test['isic_id'],pd.Series(final_pred[:,1])],axis=1)
sub_df.head()

,isic_id,0
0,ISIC_0015657,0.001925
1,ISIC_0015729,0.001900
2,ISIC_0015740,0.001950


In [139]:
from tqdm import tqdm
#train_predict = stacking_model.predict(X_train_final.drop(columns=['isic_id'],axis=1))
X_val = train_metadata_processed.drop(columns=['isic_id','target'])
y_val = train_metadata_processed['target']

y_val_pred = np.array([])
for i in tqdm(range(0,len(X_val),1000)):
    if len(X_val)-i > 1000:
        y_val_pred = np.append(y_val_pred,
                                stacking_model.predict(X_val[i:i+1000]))
    else:
        y_val_pred = np.append(y_val_pred,
                                stacking_model.predict(X_val[i:]))
    
#y_val_pred = stacking_model.predict(X_val)

100%|██████████| 402/402 [02:45<00:00,  2.43it/s]


In [147]:
sum(np.array(y_val_pred == y_val).astype(int)) / len(y_val)

fp = sum(np.array((y_val_pred==1)&(y_val==0)).astype(int)) / len(y_val)
fn = sum(np.array((y_val_pred==0)&(y_val==1)).astype(int)) / len(y_val)

fp, fn

(0.01107567714475925, 0.0)

In [91]:
#stacking_model.final_estimator?

In [37]:
#sub_df.to_csv('submission.csv', index=False)